# 🎬 Movie Recommendation using K-Means Clustering
Cluster users based on their movie ratings and recommend movies from similar users.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

## 📥 Load MovieLens 100K Dataset

In [ ]:
# Load ratings and movie info
ratings = pd.read_csv('https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv')
movies = pd.read_csv('https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/movies.csv')

# Rename columns if needed and merge
df = ratings.merge(movies, on='book_id', how='inner')
df = df[['user_id', 'book_id', 'rating', 'title']]
df.columns = ['userId', 'movieId', 'rating', 'title']

## 🧾 Create User-Movie Matrix

In [ ]:
user_movie = df.pivot_table(index='userId', columns='title', values='rating')

## 🧼 Impute and Normalize

In [ ]:
imputer = SimpleImputer(strategy='mean')
user_movie_imputed = pd.DataFrame(imputer.fit_transform(user_movie),
                                   columns=user_movie.columns,
                                   index=user_movie.index)
scaler = StandardScaler()
user_movie_scaled = scaler.fit_transform(user_movie_imputed)

## 🤖 Apply K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(user_movie_scaled)
user_movie_imputed['Cluster'] = clusters

## 🎯 Recommend Movies for a Given User

In [ ]:
def recommend_movies(user_id, n=5):
    user_cluster = user_movie_imputed.loc[user_id, 'Cluster']
    similar_users = user_movie_imputed[user_movie_imputed['Cluster'] == user_cluster].drop('Cluster', axis=1)
    mean_ratings = similar_users.mean().sort_values(ascending=False)
    user_seen = user_movie.loc[user_id].dropna().index
    recommendations = mean_ratings.drop(user_seen).head(n)
    return recommendations

# Try with a user
recommend_movies(user_id=1)

## 📊 PCA Visualization of Clusters

In [ ]:
# 🧬 PCA for Cluster Visualizationfrom sklearn.decomposition import PCApca = PCA(n_components=2)user_2d = pca.fit_transform(user_movie_scaled)cluster_df = pd.DataFrame()cluster_df['PCA1'] = user_2d[:, 0]cluster_df['PCA2'] = user_2d[:, 1]cluster_df['Cluster'] = clustersimport matplotlib.pyplot as pltimport seaborn as snsplt.figure(figsize=(10, 6))sns.scatterplot(data=cluster_df, x='PCA1', y='PCA2', hue='Cluster', palette='Set2', s=80)plt.title("🎯 Customer Segmentation based on Movie Ratings")plt.xlabel("PCA Component 1")plt.ylabel("PCA Component 2")plt.grid(True)plt.show()

## 🔍 Cluster Interpretation: What Each Segment Likes

In [ ]:
# 🧠 Cluster Interpretation: Top Rated Books per Clustercluster_labels = user_movie_imputed['Cluster']top_books_by_cluster = {}for cluster_num in range(5):    users_in_cluster = user_movie_imputed[user_movie_imputed['Cluster'] == cluster_num].drop(columns='Cluster')    mean_ratings = users_in_cluster.mean().sort_values(ascending=False).head(5)    top_books_by_cluster[cluster_num] = mean_ratings# Display top 5 books for each clusterfor cluster, books in top_books_by_cluster.items():    print(f"\n📚 Top Books for Cluster {cluster}:")    print(books)